In [1]:
from conllu_test import check_conllu_for_conditions2, head_greater_than_id, head_smaller_than_id, has_son, file_to_conllu,merge_evals, conllu_merger, check_conllu_for_conditions_v3, no_weak_pronoun, has_sons, next_head, prev_head, has_no_son, check_conds, feats_include, is_sub_clause_head_core_arg, nicht, is_not, is_not_any_of, has_no_sons, is_sub_clause_head, parent_is, merge_conditions_dicts, v_SO, v_OS

from analysis_main import eval_pipeline, conllu_file_stats, file_freqs_eval, file_freqs_eval_filtered_by_conds, conllu_file_freqs


In [2]:
# list of all the values in the conllu file generated w the function conllu_possible_values()
deprel_list = ['det', 'nsubj', 'amod', 'case', 'obj', 'flat', 'ROOT', 'mark', 'obl', 'compound', 'cc', 'conj', 'punct', 'appos', 'nmod', 'aux', 'ccomp', 'advmod', 'fixed', 'acl', 'nummod', 'cop', 'advcl', 'expl:pass', 'xcomp', 'csubj', 'iobj', 'parataxis', 'dep']
upos_list = ['DET', 'NOUN', 'ADJ', 'ADP', 'PROPN', 'VERB', 'SCONJ', 'NUM', 'CCONJ', 'PUNCT', 'AUX', 'SYM', 'PRON', 'ADV', 'INTJ', 'SPACE', 'PART']
upos_list_args= ['DET', 'NOUN', 'ADJ', 'PROPN','PRON', 'ADV','NUM', 'SYM']

#EVAL 1
# SVO in simple clauses. (sense complements verbals, sense que el verb sigui una subordinada -> pot ser coordinat)
#  we allow advcl, etc to be tere though
# 1 AQUESTA ES LA CONDICIO D UN BON VERB (sense complements verbals, sense ser xcomp d un altre verb)
# no ser el main verb d una subordinada vol dir restringir molts deprels:
# aux is also out bc it marks situations such as "cal marxar", "estan a punt d entrar"
# fixed is also aut bc it is basically "ès a dir"
# compound is also out bc it classifies constructions where verbs are very close to another "vol dir", "es fa dir"
# Evitar coordinacio amb subordinades -> per evitar frases com (1, '2025-10-10 00:33:22_habitatge_65_031_011')] de a1KI_corpus_65_habitatge_gemm.conllu
eval_v= {"verb": [{"upos": "VERB"}]}
conds_v_adequat = [{"upos": "VERB", is_not_any_of: [{"lemma":"ser"},{"deprel": is_sub_clause_head}, {"deprel": "aux"}, {"deprel": "fixed"}, {"deprel": "compound"}, {"deprel": "conj", parent_is : {"upos":"VERB", "deprel" : is_sub_clause_head} }], has_no_sons: [{"deprel": is_sub_clause_head_core_arg}, {"deprel":"cop"}]  }]
eval_v_adequat = {"verb_ok": conds_v_adequat }

# EVAL 1.1 checking the deprels of the simple verb to make sure everything is ok
eval_verb_deprels = {f"verb_{deprel}": [{"upos": "VERB", "deprel": deprel}]   for deprel in deprel_list}
eval_verb_derpel_no_subclause_arg = {f"verb_no_subclause_args_{deprel}": [{"upos": "VERB", "deprel": deprel, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}]}]   for deprel in deprel_list}

# Defining the object of a verb (clause type independent)
# eliminem pronoms febles, eliminem complements preposicionals, eliminem pronoms relatius amb funcio d objecte,
# eliminem tambe deprel case per eliminar coses com ara: "com a mesures s han aplicat.."
# eliminem tambe que l obj pugui tenir upos ADP per eliminar "donar per tancat, etc"
# possible case study: cleft sentences
conds_v_obj_good= [{"upos": "VERB", has_sons: [{"deprel": "obj", "lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos":"ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}] }] }]
conds_v_NOT_obj_good = [{"upos": "VERB", is_not_any_of: [{has_sons: [{"deprel": "obj", "lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos":"ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"},  {"deprel": "mark"}]}] }]}]
eval_obj = {"v_w_obj": conds_v_obj_good, "v_w_no_obj": conds_v_NOT_obj_good}
#"v_w_obj_und_no_obj": merge_conditions_dicts(conds_v_obj_good[0], conds_v_NOT_obj_good[0])

# Defining a proper subject of a verb (clause type independent)
# i
conds_verb_wsubj_true = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_son: {"deprel": "cop"} }] }]
conds_v_NOT_subj = [{"upos": "VERB", nicht(has_sons): [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}},has_no_son: {"deprel": "cop"} }] }]
eval_subj = {"v_subj": conds_verb_wsubj_true, "v_no_subj": conds_v_NOT_subj }
# "v_w_subj_und_no_subj": merge_conditions_dicts(conds_verb_wsubj_true[0], conds_v_NOT_subj[0])


# LIST for EVAL 1:
# this is the list of evaluations for the simple presence of S V O in "simple" clauses, this is, verbal predicates with no clausal core arguments which are also not the main verb of a subordinate clause
list_evals_presence_SVO = [eval_v, eval_v_adequat, merge_evals(eval_v_adequat, eval_subj), merge_evals(eval_v_adequat, eval_obj), merge_evals(eval_v_adequat, merge_evals(eval_subj, eval_obj)), ]


# EVAL 1.2
# what is the upos class ob our beautiful verb arguments?
# obj is not updated for upos restrictions in "good" objects (no adj, no adp)
eval_obj_upos= {f"verb__obj_{upos}": [{"upos": "VERB", has_sons: [{"deprel": "obj", "upos": upos ,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}]}]}] for upos in upos_list_args}
eval_subj_upos= {f"verb_subj_{upos}": [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "lemma": no_weak_pronoun, "upos":upos,  is_not: {feats_include: {"PronType": "Rel"}},has_no_son: {"deprel": "cop"} }] }] for upos in upos_list_args}
list_evals_SVO_type= [eval_v, eval_obj, eval_subj, eval_subj_upos, merge_evals(eval_subj, eval_obj),eval_obj_upos, merge_evals(eval_subj_upos, eval_obj_upos)]


#EVAL 2_
# order of the verb arguments in EVAL 1
# -> successful
# order of the object: OV vs VO
conds_v_obj_good_OV = [{"upos": "VERB", has_sons: [{"deprel": "obj", "head": head_greater_than_id,"lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos": "ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"},  {"deprel": "mark"}]}]}]
conds_v_obj_good_VO = [{"upos": "VERB", has_sons: [{"deprel": "obj", "head": head_smaller_than_id,"lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos": "ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}]}]}]
conds_v_obj_good_OVO = [{"upos": "VERB", has_sons: [{"deprel": "obj", "head": head_smaller_than_id,"lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos": "ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}  ]}, {"deprel": "obj", "head": head_greater_than_id,"lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos": "ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}  ]} ]}]

eval_OVO = {"obj_OV": conds_v_obj_good_OV, "obj_VO": conds_v_obj_good_VO, "obj_OVO": conds_v_obj_good_OVO}

# order of the subject
conds_v_subj_good_SV = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, "head": head_greater_than_id, has_no_son: {"deprel": "cop"} }] }]
conds_v_subj_good_VS = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, "head": head_smaller_than_id, has_no_son: {"deprel": "cop"} }] }]
conds_v_subj_good_SVS = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, "head": head_smaller_than_id, has_no_son: {"deprel": "cop"} }, {"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, "head": head_greater_than_id, has_no_son: {"deprel": "cop"} }] }]

eval_SVS= {"subj_SV": conds_v_subj_good_SV, "subj_VS": conds_v_subj_good_VS, "subj_SVS": conds_v_subj_good_SVS}

# final test: SOV vs OSV, VSO vs VOS
conds_SO = [{"head": v_SO, "lemma": no_weak_pronoun}]
conds_OS = [{"head": v_OS}]

eval_SO_order = {"SO": conds_SO, "OS": conds_OS}

list_OS_test= [eval_SO_order]

list_eval_SVO_order= list_evals_presence_SVO + [merge_evals(eval_v_adequat ,eval_OVO)] + [merge_evals(eval_v_adequat ,eval_SVS)] + [merge_evals(eval_v_adequat,merge_evals(merge_evals(eval_OVO, eval_SO_order), eval_SVS))]


# EVAL 3
# same but in subordinate clauses
# TODO: clarify if the distinction between sub itself and coordinated with a subordinate clause makes sense or not
# CONDICIONS PER UNA SUBORDINADA DE FUNCIO X
# restringim a les que tenen un nexe de tipus SCONJ, q te deprel mark -> que no fa cap funcio
list_sub_deprels = ["csubj", "ccomp", "advcl", "acl", "xcomp"]

eval_v_sub_types = {f"v_sub_deprel_{deprel}": [{"upos": "VERB", "deprel": deprel, has_son: {"upos": "SCONJ", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}

eval_v_sub_types_coord = {f"v_sub_coord_deprel_{deprel}": [{"upos": "VERB", "deprel": "conj", parent_is: {"deprel": deprel},has_son: {"upos": "SCONJ", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}

# EVAL SVO presence in subordinates and coordinates with subordinates
list_evals_SVO_presence_subordinates = [eval_v, eval_v_sub_types, merge_evals(eval_v_sub_types, eval_subj), merge_evals(eval_v_sub_types, eval_obj), merge_evals(eval_v_sub_types, merge_evals(eval_subj, eval_obj)), ]
list_evals_SVO_presence_sub_coord = [eval_v, eval_v_sub_types_coord, merge_evals(eval_v_sub_types_coord, eval_subj), merge_evals(eval_v_sub_types_coord, eval_obj), merge_evals(eval_v_sub_types_coord, merge_evals(eval_subj, eval_obj)), ]

list_evals_SVO_presence_sub_both = list_evals_SVO_presence_subordinates + list_evals_SVO_presence_sub_coord

#EVAL SVO order in subordinates and coordinates w subordinates
list_eval_SVO_order_subordinates = list_evals_SVO_presence_subordinates + [merge_evals(eval_v_sub_types ,eval_OVO)] + [merge_evals(eval_v_sub_types ,eval_SVS)] + [merge_evals(eval_v_sub_types, merge_evals(eval_OVO, eval_SVS))]
list_eval_SVO_order_sub_coord = list_evals_SVO_presence_sub_coord + [merge_evals(eval_v_sub_types_coord ,eval_OVO)] + [merge_evals(eval_v_sub_types_coord ,eval_SVS)] + [merge_evals(eval_v_sub_types_coord, merge_evals(eval_OVO, eval_SVS))]

list_evals_SVO_order_sub_both = list_eval_SVO_order_subordinates + list_eval_SVO_order_sub_coord

# EVAL 3.5
# tipus de subordinades mes comprimit
# en comptes de tenir un nexe "SCONJ" (que, si,) que passa amb les q tenen nexe "ADP" (cansats de compartir, per destituir el president..)
# i a mes el verb esta en infinitiu:
eval_v_sub_types3 = {f"v_sub2_infinitiu_deprel_{deprel}": [{"upos": "VERB", feats_include: {'VerbForm':'Inf'}, "deprel": deprel, has_son: {"upos": "ADP", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}
eval_v_sub_types3_coord = {f"v_sub_infinitiu_coord_deprel_{deprel}": [{"upos": "VERB", feats_include: {'VerbForm':'Inf'}, "deprel": "conj", parent_is: {"deprel": deprel},has_son: {"upos": "ADP", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}
list_evals_subs_SVO_presence = [eval_v_sub_types3, eval_v_sub_types3_coord, merge_evals(eval_v_sub_types3, eval_obj), merge_evals(eval_v_sub_types3, eval_subj)]


conds_v_obj_OV_adj = [{"upos": "VERB", has_sons: [{"deprel": "obj", "upos":"ADJ","head": head_greater_than_id,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "mark"}]}]}]
conds_v_obj_VO_adj = [{"upos": "VERB", has_sons: [{"deprel": "obj", "upos":"ADJ", "head": head_smaller_than_id,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "mark"}]}]}]


eval_obj_adj_problem = {"OV_adj": conds_v_obj_OV_adj , "VO_not_adj": conds_v_obj_VO_adj}
list_OVO_adj_problem = [merge_evals(eval_v_adequat, eval_obj_adj_problem)]


# EVAL 4
# NOUN PHRASE WHATS UP
# idea: upos based
# nur nominale sintagmas die nur nomen haben

nom_deprels = ["nmod","appos", "nummod","acl", "amod", "det", "clf", "case"]
#general noun conditions: maybe i add more
conds_noun_ok= [{"upos":"NOUN"}]
eval_noun= {"eval_n": conds_noun_ok}

# DET ORDER
conds_n_det = [{"upos": "NOUN", has_sons: [{"upos": "DET", "head": head_smaller_than_id, is_not: {"deprel":"conj"}, has_no_sons: [{"deprel": "mark"}, {"deprel": "appos"}]}]}]
#  det_n doesnt have the same problems that the characterisation of n_det has
conds_det_n= [{"upos": "NOUN", has_sons: [  {"upos": "DET", "head": head_greater_than_id, is_not: {"deprel":"conj"}, has_no_sons: [{"deprel": "mark"}, {"deprel": "appos"}]}] }]

conds_det_n_det = [merge_conditions_dicts(conds_det_n[0], conds_n_det[0])]

eval_det_n = {"det_n": conds_det_n, "n_det":conds_n_det,  "det_n_det":conds_det_n_det}

# ADJ + N order
conds_adj_n = [{"upos": "NOUN", has_sons: [{"upos": "ADJ", "head": head_greater_than_id, is_not_any_of: [{"deprel": "cop"}, {"deprel": "acl"}]}]}]
conds_n_adj =[{"upos": "NOUN", has_sons: [{"upos": "ADJ", "head": head_smaller_than_id, is_not_any_of: [{"deprel": "cop"}, {"deprel": "acl"}]}]}]
conds_adj_n_adj= merge_conditions_dicts(conds_n_adj[0], conds_adj_n[0])

eval_adj_n = {"adj_n": conds_adj_n, "n_adj":conds_n_adj, "adj_n_adj":[conds_adj_n_adj]}

# ADV + ADJ
conds_adv_adj_n = [{"upos": "NOUN", has_sons: [{"upos": "ADJ", "head": head_greater_than_id,  has_sons: [{"upos": "ADV"}] }]}]
conds_n_adv_adj =[{"upos": "NOUN", has_sons: [{"upos": "ADJ", "head": head_smaller_than_id, has_sons: [{"upos": "ADV"}] }]}]
conds_adv_adj_n_adj= [merge_conditions_dicts(conds_n_adv_adj[0], conds_adv_adj_n[0])]

eval_adv_adj_n = {"adv_adj_n": conds_adv_adj_n, "n_adv_adj": conds_n_adv_adj, "n_adv_adj_beide":conds_adv_adj_n_adj}


list_eval_adjs= [eval_noun, eval_det_n, eval_adj_n, eval_adv_adj_n, merge_evals(eval_det_n, eval_adj_n), merge_evals(eval_det_n, eval_adv_adj_n)]
# EVAL 5:
# numerals
conds_n_num= [{"upos": "NOUN", has_sons: [{"deprel": "nummod"}]}]
conds_n_num_upos= [{"upos": "NOUN", has_sons: [{"upos": "NUM"}]}]
conds_n_num2 = [{"upos":"NOUN", has_sons: [{"upos": "NUM", is_not: {"deprel": "nummod"} } ] }]
conds_n_num3 = [{"upos":"NOUN", has_sons: [{"deprel": "nummod", is_not: {"upos": "NUM"} } ] }]

eval_n_num ={"n_nummod": conds_n_num, "n_num_upos": conds_n_num_upos, "num_not_nummod": conds_n_num2, "nummod_not_num": conds_n_num3 }

list_nom_evals= [eval_noun, eval_det_n, eval_adj_n, eval_adv_adj_n, merge_evals(eval_det_n, eval_adj_n), merge_evals(eval_det_n, eval_adv_adj_n), ]


# EVAL: feats dels adverbis q acompanyen adjectius
conds_adv_adj= [{"upos": "ADV", parent_is: {"upos": "ADJ", "head": head_greater_than_id, parent_is: {"upos": "NOUN"}}}]
conds_n_adv_adj= [{"upos": "ADV", parent_is: {"upos": "ADJ", "head": head_smaller_than_id, parent_is: {"upos": "NOUN"}}}]

eval_adv_adj = {"adv_adj_n": conds_adv_adj, "n_adv_adj": conds_n_adv_adj}
adv_possible_feats= [{'Polarity': 'Neg'},{'Degree': 'Cmp'}]

eval_adv_types= {f"feat{feature}": [{feats_include: feature}] for feature in adv_possible_feats}
llista_adv_types = [eval_adv_adj, merge_evals(eval_adv_types, eval_adv_adj)]





In [3]:
codi = "b6"
b6_list= ['b6KI_corpus_68_estatsunits_gemm.conllu', 'b6KI_corpus_70_incendis_gemm.conllu', 'b6KI_corpus_70_migracions_gemm.conllu', 'b6KI_corpus_69_educacio_gemm.conllu', 'b6KI_corpus_66_musica_gemm.conllu', 'b6KI_corpus_70_salut_gemm.conllu', 'b6KI_corpus_68_erc_gemm.conllu', 'b6KI_corpus_69_meteorologia_gemm.conllu', 'b6KI_corpus_67_psoe_gemm.conllu', 'b6KI_corpus_68_habitatge_gemm.conllu', 'b6KI_corpus_15_russia_gemm.conllu', 'b6KI_corpus_33_israel_gemm.conllu', 'b6KI_corpus_7_ucraina_gemm.conllu']+['b6tv3_corpus_70_estatsunits.conllu', 'b6tv3_corpus_70_incendis.conllu', 'b6tv3_corpus_70_migracions.conllu', 'b6tv3_corpus_70_educacio.conllu', 'b6tv3_corpus_67_musica.conllu', 'b6tv3_corpus_70_salut.conllu', 'b6tv3_corpus_69_erc.conllu', 'b6tv3_corpus_70_meteorologia.conllu', 'b6tv3_corpus_70_psoe.conllu', 'b6tv3_corpus_70_habitatge.conllu', 'b6tv3_corpus_15_russia.conllu', 'b6tv3_corpus_33_israel.conllu', 'b6tv3_corpus_7_ucraina.conllu']+[f"{codi}_KI_corpus.conllu", f"{codi}_tv3_corpus.conllu"]

 #corpus
tv3_corpus=f"{codi}_tv3_corpus.conllu"
KI_corpus=f"{codi}_KI_corpus.conllu"

tv3_small= f"{codi}tv3_corpus_14_russia.conllu"
KI_small= f"{codi}KI_corpus_14_russia_gemm.conllu"
tv3_mid= f"{codi}tv3_corpus_70_incendis.conllu"
KI_mid= f"{codi}KI_corpus_70_incendis_gemm.conllu"

mini = "mini.conllu"
#some handy corpus lists:
file_list=[tv3_corpus, KI_corpus]
small_file_list=[tv3_small, KI_small]
mid_file_list=[tv3_mid, KI_mid]

In [4]:
mega_SVO_eval_list = list_evals_SVO_order_sub_both+list_eval_SVO_order
#took 2min for the mid list
# SVO order tkaes 5 min

eval_pipeline(list_nom_evals, file_list, f"{codi}_big_nom_evals.csv")

Evaluating list 1 of 6 on b6_tv3_corpus.conllu
	Evaluating eval_n on b6_tv3_corpus.conllu
	 	Evaluated eval_n on b6_tv3_corpus.conllu, it took 3.818284273147583
done with list, it took 3.818375587463379
Evaluating list 2 of 6 on b6_tv3_corpus.conllu
	Evaluating det_n on b6_tv3_corpus.conllu
	 	Evaluated det_n on b6_tv3_corpus.conllu, it took 4.5823750495910645
	Evaluating n_det on b6_tv3_corpus.conllu
	 	Evaluated n_det on b6_tv3_corpus.conllu, it took 4.938475131988525
	Evaluating det_n_det on b6_tv3_corpus.conllu
	 	Evaluated det_n_det on b6_tv3_corpus.conllu, it took 8.875991106033325
done with list, it took 18.397104501724243
Evaluating list 3 of 6 on b6_tv3_corpus.conllu
	Evaluating adj_n on b6_tv3_corpus.conllu
	 	Evaluated adj_n on b6_tv3_corpus.conllu, it took 4.316155672073364
	Evaluating n_adj on b6_tv3_corpus.conllu
	 	Evaluated n_adj on b6_tv3_corpus.conllu, it took 4.429828405380249
	Evaluating adj_n_adj on b6_tv3_corpus.conllu
	 	Evaluated adj_n_adj on b6_tv3_corpus.conll

0

In [5]:
# codeblock to merge all files into one fat file :)
import glob
llista=["estatsunits","incendis","migracions", "educacio", "musica", "salut", "erc", "meteorologia", "psoe", "habitatge", "russia", "israel", "ucraina"]
type= "tv3"
llista_arxius=[]
for tema in llista:
    print("adding", tema)
    if not glob.glob(f"analysed_corpus/{codi}{type}_corpus_*_{tema}*.conllu"):
        raise Exception(f"File for topic {codi}{type}_corpus_*_{tema}*.conllu not found.")
    elif len(glob.glob(f"analysed_corpus/{codi}{type}_corpus_*_{tema}*.conllu")) > 0:
        glob.glob(f"analysed_corpus/{codi}{type}_corpus_*_{tema}*.conllu").sort(reverse=True)
        corpuspath = glob.glob(f"analysed_corpus/{codi}{type}_corpus_*_{tema}*.conllu")[0]
        corpusfile = corpuspath.split("/")[-1]
    llista_arxius.append(corpusfile)
conllu_merger(llista_arxius, f"{codi}_{type}_corpus.conllu")



adding estatsunits
adding incendis
adding migracions
adding educacio
adding musica
adding salut
adding erc
adding meteorologia
adding psoe
adding habitatge
adding russia
adding israel
adding ucraina


In [18]:
print(llista_arxius)

#eval_pipeline(llista_adv_types, mid_file_list, "mid_adv_types_problem.csv")

['b6tv3_corpus_70_estatsunits.conllu', 'b6tv3_corpus_70_incendis.conllu', 'b6tv3_corpus_70_migracions.conllu', 'b6tv3_corpus_70_educacio.conllu', 'b6tv3_corpus_67_musica.conllu', 'b6tv3_corpus_70_salut.conllu', 'b6tv3_corpus_69_erc.conllu', 'b6tv3_corpus_70_meteorologia.conllu', 'b6tv3_corpus_70_psoe.conllu', 'b6tv3_corpus_70_habitatge.conllu', 'b6tv3_corpus_15_russia.conllu', 'b6tv3_corpus_33_israel.conllu', 'b6tv3_corpus_7_ucraina.conllu']


In [9]:

conllu_file_stats(tv3_corpus)

{'words': 221946,
 'sentences': 8427,
 'av. sentence length': 26.3374866500534,
 'articles': 757,
 'sentences/article': 11.132100396301189,
 'av. tree height': 4.820220719117123,
 'max tree height': 14,
 'tree height / sent length': 0.20534697753223818}

In [10]:
#todo: check if the double article length really makes sense oh god i cry
conllu_file_stats(KI_corpus)

{'words': 148464,
 'sentences': 5985,
 'av. sentence length': 24.806015037593983,
 'articles': 718,
 'sentences/article': 8.335654596100278,
 'av. tree height': 4.829908103592314,
 'max tree height': 13,
 'tree height / sent length': 0.21537266651076792}